# Example: Validation Report and Compliance Config

In this example, we evaluate the Session 3 engine variants against four formal deployment gates, produce a **validation report** with pass/fail verdicts, and export the **compliance configuration** that Session 4's production system enforces in real time.

The first two S3 examples produced four candidate strategies. This example asks the production question: which of them is good enough to deploy with real money? The answer comes from formal pass/fail gates, not from eyeballing a scorecard, and the same numerical thresholds become the compliance config that Session 4's live system enforces in real time.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
> * __Apply the four validation gates:__ Evaluate each strategy (EWLS engine, bandit-learned elasticity, heuristic elasticity, frozen baseline) against Sharpe, drawdown, failure rate, and benchmark thresholds. Produce a pass/fail table that shows whether each candidate clears the four deployment gates.
> * __Export compliance configuration:__ Define pre-approved operating limits (concentration cap, drawdown gate, turnover limit, position size limit) and save them for Session 4. The exported config is what the live execution system loads to decide which trades auto-execute and which queue for human review.
> * __Produce a strategy comparison dashboard:__ Visualize the distributional performance of all strategies side by side using box plots and histograms. The dashboard exposes spread and tails that the median scorecard hides.

This is the gate between simulation and production.

___

## Setup, Data and Prerequisites
We begin by loading our packages and helper functions via [the `Include.jl` file](./Include.jl). This activates the local [Julia](https://julialang.org) environment and loads all dependencies.

In [ ]:
# --- Load packages and helper functions ---
include("Include.jl"); # The Include.jl file activates the local Julia environment and imports all dependencies.

### Constants
In the section below, we set some constants that will be used throughout the notebook. We can modify these constants to explore different scenarios and allocations.

See the comments in the code for more details on each constant, its purpose, units, etc.

In [ ]:
# --- Validation inputs (Examples 1 and 2 hand-offs) ---
EWLS_RESULTS_PATH = joinpath(_PATH_TO_DATA, "ewls-replay-results.jld2")        # per-path metrics for the frozen and EWLS engines
ETA_BANDIT_RESULTS_PATH = joinpath(_PATH_TO_DATA, "eta-bandit-results.jld2")   # per-path metrics for bandit-learned and heuristic CES
B₀ = 10_000.0                                  # starting budget (USD); used to compute the failure rate

# --- Validation gates (deployment thresholds applied in Task 1) ---
VALIDATION_MIN_SHARPE = 0.3                    # minimum acceptable median Sharpe ratio
VALIDATION_MAX_DRAWDOWN = 0.25                 # maximum acceptable median drawdown (fraction)
VALIDATION_MAX_FAILURE_RATE = 0.10             # max acceptable fraction of paths ending below B₀

# --- Compliance limits (exported to Session 4) ---
COMPLIANCE_CONCENTRATION_CAP   = 0.40          # max single-asset weight in the live portfolio (fraction)
COMPLIANCE_DRAWDOWN_GATE       = 0.15          # max drawdown before the engine de-risks to cash (fraction)
COMPLIANCE_TURNOVER_LIMIT      = 0.50          # max fraction of wealth traded per rebalance
COMPLIANCE_POSITION_SIZE_LIMIT = 5_000.0       # max dollar exposure per individual trade (USD)

Load the backtest results from Examples 1 and 2. The EWLS replay results provide the frozen and online engine distributions; the elasticity bandit results provide the bandit-learned and heuristic CES distributions. Both hand-offs include per-path NPV arrays so we can apply the NPV-vs-RF gate without recomputing the discount factor here.

In the code block below, we load `ewls-replay-results.jld2` (Example 1 hand-off) and `eta-bandit-results.jld2` (Example 2 hand-off), unpack the per-path metric arrays for each strategy, and bind them to top-level globals.

The cell returns:

* `frozen_fw::Vector{Float64}`, `frozen_dd::Vector{Float64}`, `frozen_sharpe::Vector{Float64}`, `frozen_npv::Vector{Float64}`: per-path terminal wealth, max drawdown, Sharpe, and NPV (vs continuously-compounded risk-free) for the frozen Cobb-Douglas engine.
* `ewls_fw::Vector{Float64}`, `ewls_dd::Vector{Float64}`, `ewls_sharpe::Vector{Float64}`, `ewls_npv::Vector{Float64}`: per-path metrics for the EWLS online-update engine ($h = 63$).
* `bandit_fw::Vector{Float64}`, `bandit_dd::Vector{Float64}`, `bandit_sharpe::Vector{Float64}`, `bandit_npv::Vector{Float64}`: per-path metrics for the bandit-learned elasticity engine.
* `heur_fw::Vector{Float64}`, `heur_dd::Vector{Float64}`, `heur_sharpe::Vector{Float64}`, `heur_npv::Vector{Float64}`: per-path metrics for the heuristic $\eta(\lambda)$ engine.
* `n_paths::Int`: number of Monte Carlo paths shared across all strategies.

In [ ]:
(; frozen_fw, frozen_dd, frozen_sharpe, frozen_npv,
   ewls_fw, ewls_dd, ewls_sharpe, ewls_npv, n_paths,
   bandit_fw, bandit_dd, bandit_sharpe, bandit_npv,
   heur_fw, heur_dd, heur_sharpe, heur_npv) = let
    # --- Step 1: Load EWLS results from Example 1 ---
    ewls_data = load_results(EWLS_RESULTS_PATH);
    frozen_fw     = ewls_data["frozen_final_wealth"];
    frozen_dd     = ewls_data["frozen_max_dd"];
    frozen_sharpe = ewls_data["frozen_sharpe"];
    frozen_npv    = ewls_data["frozen_npv"];
    ewls_fw       = ewls_data["ewls_final_wealth"];
    ewls_dd       = ewls_data["ewls_max_dd"];
    ewls_sharpe   = ewls_data["ewls_sharpe"];
    ewls_npv      = ewls_data["ewls_npv"];
    n_paths       = ewls_data["n_paths"];

    # --- Step 2: Load eta-bandit results from Example 2 ---
    bandit_data = load_results(ETA_BANDIT_RESULTS_PATH);
    bandit_fw     = bandit_data["bandit_final_wealth"];
    bandit_dd     = bandit_data["bandit_max_dd"];
    bandit_sharpe = bandit_data["bandit_sharpe"];
    bandit_npv    = bandit_data["bandit_npv"];
    heur_fw       = bandit_data["heur_final_wealth"];
    heur_dd       = bandit_data["heur_max_dd"];
    heur_sharpe   = bandit_data["heur_sharpe"];
    heur_npv      = bandit_data["heur_npv"];

    println("Loaded results: $(n_paths) paths, 4 strategies")
    (frozen_fw = frozen_fw, frozen_dd = frozen_dd, frozen_sharpe = frozen_sharpe, frozen_npv = frozen_npv,
     ewls_fw = ewls_fw, ewls_dd = ewls_dd, ewls_sharpe = ewls_sharpe, ewls_npv = ewls_npv, n_paths = n_paths,
     bandit_fw = bandit_fw, bandit_dd = bandit_dd, bandit_sharpe = bandit_sharpe, bandit_npv = bandit_npv,
     heur_fw = heur_fw, heur_dd = heur_dd, heur_sharpe = heur_sharpe, heur_npv = heur_npv)
end

___
## Task 1: Apply the Five Validation Gates
In this task, we evaluate each strategy against the deployment criteria: median Sharpe $\geq$ 0.3, median max drawdown $\leq$ 25%, failure rate $\leq$ 10%, beats the frozen baseline on median wealth, and median NPV vs the continuously-compounded risk-free baseline $\geq$ 0. The frozen Cobb-Douglas engine plays the role of the reference benchmark on the wealth gate; the NPV gate uses the risk-free rate as the universal opportunity-cost benchmark. The other three strategies must clear all five gates to qualify for deployment.

> __What should we see?__
>
> The EWLS and bandit-elasticity strategies should pass most or all gates. The frozen baseline may fail on Sharpe or drawdown if it cannot adapt to regime shifts; by construction it satisfies the wealth-vs-frozen gate by equality. The NPV gate distinguishes _the typical scenario beats cash_ from _the typical scenario beats a peer strategy_ — a strategy can pass the wealth-vs-frozen gate while still failing the NPV gate if the frozen baseline itself underperformed risk-free.

The code block below returns `reports::Vector{`[`MyValidationReport`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.MyValidationReport)`}`, one report per strategy, each carrying the per-gate `passed::Dict{String,Bool}`, actual metric values, and threshold values.

In [ ]:
reports = let
    # --- Step 1: Define validation criteria (5 gates: Sharpe, DD, fail rate, wealth-vs-frozen, NPV-vs-RF) ---
    frozen_median_wealth = median(frozen_fw);

    criteria = Dict(
        "min_sharpe" => VALIDATION_MIN_SHARPE,
        "max_drawdown" => VALIDATION_MAX_DRAWDOWN,
        "max_failure_rate" => VALIDATION_MAX_FAILURE_RATE,
        "min_wealth_vs_frozen" => frozen_median_wealth,
        "min_median_npv" => 0.0,
    );

    # --- Step 2: Build validation reports ---
    strategies = [
        ("EWLS Engine (h=63)", ewls_fw, ewls_dd, ewls_sharpe, ewls_npv),
        ("Bandit-η CES",       bandit_fw, bandit_dd, bandit_sharpe, bandit_npv),
        ("Heuristic-η CES",    heur_fw, heur_dd, heur_sharpe, heur_npv),
        ("Frozen CD Engine",   frozen_fw, frozen_dd, frozen_sharpe, frozen_npv),
    ];

    reports = MyValidationReport[];
    for (label, fw, dd, sr, npv) in strategies
        actuals = Dict(
            "min_sharpe" => median(sr),
            "max_drawdown" => median(dd),
            "max_failure_rate" => mean(fw .< B₀),
            "min_wealth_vs_frozen" => median(fw),
            "min_median_npv" => median(npv),
        );
        report = build(MyValidationReport;
            strategy_label = label, criteria = criteria, actuals = actuals);
        push!(reports, report);
    end

    # --- Step 3: Build wide-form pass/fail DataFrame (one row per strategy) ---
    # Local cell-formatter: "✓ (actual op threshold)" or "✗ (actual op threshold)"
    fmt_cell = (r, k, op) -> "$(r.passed[k] ? "✓" : "✗") ($(round(r.actuals[k], digits=3)) $(op) $(round(r.criteria[k], digits=3)))"

    report_df = DataFrame(
        "Strategy"          => [r.strategy_label for r in reports],
        "Sharpe"            => [fmt_cell(r, "min_sharpe", "≥") for r in reports],
        "Drawdown"          => [fmt_cell(r, "max_drawdown", "≤") for r in reports],
        "Failure rate"      => [fmt_cell(r, "max_failure_rate", "≤") for r in reports],
        "Wealth vs frozen"  => [fmt_cell(r, "min_wealth_vs_frozen", "≥") for r in reports],
        "NPV vs RF"         => [fmt_cell(r, "min_median_npv", "≥") for r in reports],
        "Verdict"           => [all(values(r.passed)) ? "PASS" : "FAIL" for r in reports],
    );
    println("Validation report: pass/fail deployment gates (5 gates including NPV-vs-RF)")
    pretty_table(report_df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))
    reports
end

___
## Task 2: Export Compliance Configuration
In this task, we define the pre-approved operating limits for Session 4's production system. These limits determine which trades auto-execute and which queue for human review.

> __What should we see?__
>
> A dictionary of compliance parameters saved to disk. These values come from the validation analysis: the drawdown gate matches the engine's trigger rule, the concentration cap limits single-asset exposure, and the turnover limit bounds daily trading activity.

The code block below returns `compliance_config::Dict{String,Any}` carrying the four pre-approved operating limits that Session 4's production system enforces in real time. The configuration is built via [`build_compliance_config(...)`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.build_compliance_config).

In [ ]:
compliance_config = let
    # --- Step 1: Build compliance config ---
    compliance_config = build_compliance_config(;
        concentration_cap = COMPLIANCE_CONCENTRATION_CAP,
        drawdown_gate = COMPLIANCE_DRAWDOWN_GATE,
        turnover_limit = COMPLIANCE_TURNOVER_LIMIT,
        position_size_limit = COMPLIANCE_POSITION_SIZE_LIMIT
    );

    # --- Step 2: Display ---
    println("="^60)
    println("  COMPLIANCE CONFIGURATION FOR SESSION 4")
    println("="^60)
    for (k, v) in sort(collect(compliance_config))
        println("  $(k): $(v)")
    end
    println("\nTrades within these limits auto-execute.")
    println("Trades exceeding any limit queue for human review.")

    # --- Step 3: Save for Session 4 ---
    save_results(joinpath(_PATH_TO_DATA, "compliance-config.jld2"), compliance_config);
    println("\nSaved to compliance-config.jld2")
    compliance_config
end

___
## Task 3: Strategy Comparison Dashboard
In this task, we visualize the distributional performance of all strategies side by side. Box plots show the spread of terminal wealth and maximum drawdown; histograms show the Sharpe ratio distribution.

> __What should we see?__
>
> Strategies that passed validation should have higher median wealth, lower drawdown spread, and a right-shifted Sharpe distribution compared to those that failed.

In the code block below, we render a three-panel figure: a terminal-wealth box plot with the breakeven line, a max-drawdown box plot with the 25% gate marked, and an overlaid Sharpe histogram with the 0.3 gate marked.

In [ ]:
let
    labels = ["EWLS" "Bandit-η" "Heuristic-η" "Frozen"];

    # --- Panel 1: Terminal wealth box plot ---
    wealth_data = hcat(ewls_fw ./ B₀, bandit_fw ./ B₀, heur_fw ./ B₀, frozen_fw ./ B₀);
    p1 = boxplot(labels, wealth_data, legend=false, ylabel="W/W₀",
        title="Terminal Wealth Distribution", color=:steelblue, alpha=0.7);
    hline!(p1, [1.0], linestyle=:dash, color=:red, label="");

    # --- Panel 2: Max drawdown box plot ---
    dd_data = hcat(ewls_dd .* 100, bandit_dd .* 100, heur_dd .* 100, frozen_dd .* 100);
    p2 = boxplot(labels, dd_data, legend=false, ylabel="Max Drawdown (%)",
        title="Drawdown Distribution", color=:coral, alpha=0.7);
    hline!(p2, [25.0], linestyle=:dash, color=:red, label="");

    # --- Panel 3: Sharpe histogram overlay ---
    p3 = histogram(ewls_sharpe, bins=40, alpha=0.5, label="EWLS", color=:steelblue);
    histogram!(p3, bandit_sharpe, bins=40, alpha=0.5, label="Bandit-η", color=:goldenrod);
    histogram!(p3, frozen_sharpe, bins=40, alpha=0.3, label="Frozen", color=:gray50);
    vline!(p3, [0.3], linestyle=:dash, color=:red, label="Gate");
    xlabel!(p3, "Sharpe Ratio"); ylabel!(p3, "Count");
    title!(p3, "Sharpe Distribution");

    display(plot(p1, p2, p3, layout=(1,3), size=(1200, 400)))
end;

___
### Reading These Numbers Honestly

Our validation compared four pre-specified strategies against four pre-specified gates. That discipline matters, but the gap between _pre-specified_ and _one of many things we tried_ is where backtests get misread.

[Harvey, Liu, and Zhu (2016)](https://doi.org/10.1093/rfs/hhv059) reviewed roughly 300 published equity factors and argued that most do not survive honest multiple-testing correction. __Prof. Chau's lecture in module 1 of this course__ frames the same point at the strategy level: 1000 strategies tried on random data will produce dozens that clear $p = 0.05$ and a handful with annualized Sharpe above 1.0, all of them pure noise. The cell below reproduces the empirical point: we simulate 1000 random strategies with zero true mean return and count how many clear the usual thresholds.


In [ ]:
let
    # --- Step 1: Simulation parameters ---
    Random.seed!(42);
    n_strategies = 1000;
    n_years      = 5;
    n_days       = n_years * 252;
    sigma_daily  = 0.01;

    # --- Step 2: Simulate 1000 independent white-noise strategies ---
    sharpes = zeros(n_strategies);
    tstats  = zeros(n_strategies);
    for s ∈ 1:n_strategies
        r           = sigma_daily .* randn(n_days);
        sample_mean = mean(r);
        sample_std  = std(r);
        sharpes[s]  = (sample_mean / sample_std) * sqrt(252);
        tstats[s]   = sample_mean / (sample_std / sqrt(n_days));
    end

    # --- Step 3: Count false positives under the null ---
    pass_p05      = count(abs.(tstats) .> 1.96);
    pass_sharpe_1 = count(sharpes .> 1.0);
    best_sharpe   = maximum(sharpes);

    # --- Step 4: Null-distribution summary table ---
    null_df = DataFrame(
        "Metric" => [
            "Strategies tested",
            "Years of daily data per strategy",
            "Strategies with |t| > 1.96 (p < 0.05 two-sided)",
            "Strategies with annualized Sharpe > 1.0",
            "Best Sharpe observed under the null",
        ],
        "Value" => Any[
            n_strategies,
            n_years,
            "$(pass_p05) / $(n_strategies)  (~$(round(100*pass_p05/n_strategies, digits=1))%)",
            "$(pass_sharpe_1) / $(n_strategies)  (~$(round(100*pass_sharpe_1/n_strategies, digits=1))%)",
            round(best_sharpe, digits = 2),
        ],
    );
    println("Null distribution: $(n_strategies) random strategies, $(n_years) years of daily data, zero true mean return.")
    pretty_table(null_df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))

    # --- Panel 1: Sharpe histogram under the null ---
    p = histogram(sharpes; bins=50, color=:gray70, lc=:gray30, alpha=0.8,
        xlabel="Annualized Sharpe Ratio (null: zero true mean)", ylabel="Strategy count",
        label="Random strategies under H₀",
        title="$(n_strategies) white-noise strategies, $(n_years) years of daily data",
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent);
    vline!(p, [1.0]; lw=2.5, color=:red, ls=:dash, label="Sharpe = 1.0");
    display(p)
end;

Roughly 5% of random strategies clear standard significance by construction of the test, and with 1000 candidates tried we see Sharpe values above 1.0 that are pure sampling noise. Our four-strategy comparison above has a fixed hypothesis and a pre-registered gate list, which narrows this risk considerably. The lesson is that any extension of the comparison (a fifth strategy, a re-tuned threshold, a second run on different seeds) inflates the false-positive budget and should be tracked in the validation log, not absorbed silently.
___


___
## Summary
This example applied four formal validation gates to four strategies, produced pass/fail verdicts, exported a compliance configuration for Session 4, and visualized the distributional comparison. The compliance config is saved to `compliance-config.jld2`.

> __Key Takeaways:__
>
> * __Validation gates produce a binary deployment decision:__ Each strategy either passes all four criteria or fails. This replaces subjective judgment with an explicit, auditable process.
> * __Compliance parameters bridge simulation to production:__ The concentration cap, drawdown gate, turnover limit, and position size limit become hard constraints in Session 4's execution system. The same config that approves a backtest is the one the live system loads.
> * __The dashboard makes distributional differences visible:__ Box plots and histograms reveal not just median performance but the full spread of outcomes. Tail risks that summary statistics hide become readable as visible spread, skew, and outliers.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.